In [1]:
# =============================================================================
# VALIDATION DE L'AGENT SAC (PortfolioSACAgent)
# =============================================================================
# === CELLULE 1: IMPORTS ET CONFIGURATION ===
import os
import sys
import pickle
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("--- Initialisation du banc d'essai pour PortfolioSACAgent ---")

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

try:
    from src.agent import PortfolioSACAgent, SACActor, SACCritic
    from src.replay_buffer import ReplayBuffer
    print("✅ Importations réussies.")
except ImportError as e:
    print(f"❌ ERREUR D'IMPORTATION: {e}")
    raise

# === CELLULE 2: DÉFINITION DES PARAMÈTRES ===
print("\n--- Définition des paramètres de test ---")

# Chargement des données pour obtenir les dimensions
base_path = '..'
processed_data_path = os.path.join(base_path, 'processed_data')

try:
    with open(os.path.join(processed_data_path, 'all_data.pkl'), 'rb') as f:
        all_data = pickle.load(f)
    with open(os.path.join(processed_data_path, 'fundamentals_panel.pkl'), 'rb') as f:
        fundamentals_panel = pickle.load(f)
except Exception as e:
    print(f"❌ ERREUR: {e}")
    raise

# Paramètres pour l'agent
K_ASSETS = 10
N_TOTAL_ASSETS = len(all_data.columns.get_level_values(0).unique())
N_INDICATORS = 20  # Nombre d'indicateurs techniques
N_FUNDAMENTALS = fundamentals_panel.shape[1]  # Nombre de features fondamentales

STATE_DIM = N_TOTAL_ASSETS * 3 + 2 + K_ASSETS * (N_INDICATORS + N_FUNDAMENTALS)
ACTION_DIM = K_ASSETS

print(f"""
Paramètres calculés:
- Nombre total d'actifs: {N_TOTAL_ASSETS}
- Nombre d'indicateurs: {N_INDICATORS}
- Nombre de fondamentaux: {N_FUNDAMENTALS}
- Dimension de l'état: {STATE_DIM}
- Dimension de l'action: {ACTION_DIM}
""")

# === CELLULE 3: CRÉATION DE L'AGENT ===
print("\n--- Création de l'agent SAC ---")

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Utilisation du device: {device}")

try:
    agent = PortfolioSACAgent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        k_assets=K_ASSETS,
        n_total_assets=N_TOTAL_ASSETS,
        n_indicators=N_INDICATORS,
        n_fundamentals=N_FUNDAMENTALS,
        device=device,
        seed=42
    )
    print("✅ Agent SAC créé avec succès!")
except Exception as e:
    print(f"❌ ERREUR lors de la création de l'agent: {e}")
    raise

# === CELLULE 4: VALIDATION DE L'ARCHITECTURE ===
print("\n" + "="*60)
print("🧪 TEST 1: VALIDATION DE L'ARCHITECTURE")
print("="*60)

success = agent.validate_architecture()
if success:
    print("✅ Architecture validée avec succès!")
else:
    print("❌ Échec de la validation de l'architecture.")

# === CELLULE 5: TEST DE LA SÉLECTION D'ACTION ===
print("\n" + "="*60)
print("🧪 TEST 2: SÉLECTION D'ACTION")
print("="*60)

# Création d'une observation factice
dummy_state = np.random.randn(STATE_DIM).astype(np.float32)

# Test en mode stochastique
stochastic_action = agent.select_action(dummy_state, deterministic=False)
print(f"\nAction stochastique:")
print(f"   - Shape: {stochastic_action.shape}")
print(f"   - Somme: {np.sum(stochastic_action):.6f}")
print(f"   - Exemple: {np.round(stochastic_action, 3)}")

# Test en mode déterministe
deterministic_action = agent.select_action(dummy_state, deterministic=True)
print(f"\nAction déterministe:")
print(f"   - Shape: {deterministic_action.shape}")
print(f"   - Somme: {np.sum(deterministic_action):.6f}")
print(f"   - Exemple: {np.round(deterministic_action, 3)}")

# === CELLULE 6: TEST DU REPLAY BUFFER ===
print("\n" + "="*60)
print("🧪 TEST 3: REPLAY BUFFER")
print("="*60)

buffer = ReplayBuffer(capacity=1000, seed=42)

# Ajout de transitions factices
for i in range(10):
    state = np.random.randn(STATE_DIM)
    action = np.random.rand(ACTION_DIM)
    reward = np.random.randn()
    next_state = np.random.randn(STATE_DIM)
    done = np.random.choice([True, False])
    buffer.push(state, action, reward, next_state, done)

print(f"✅ {len(buffer)} transitions ajoutées au buffer.")

# Test de l'échantillonnage
batch = buffer.sample(5)
if batch is not None:
    states, actions, rewards, next_states, dones = batch
    print(f"✅ Échantillonnage réussi:")
    print(f"   - Shape des états: {states.shape}")
    print(f"   - Shape des actions: {actions.shape}")
    print(f"   - Shape des récompenses: {rewards.shape}")
else:
    print("❌ Échec de l'échantillonnage.")

# === CELLULE 7: TEST DE LA MISE À JOUR DE L'AGENT ===
print("\n" + "="*60)
print("🧪 TEST 4: MISE À JOUR DE L'AGENT")
print("="*60)

# Remplissage du buffer avec des données factices
for _ in range(100):
    state = np.random.randn(STATE_DIM)
    action = np.random.rand(ACTION_DIM)
    reward = np.random.randn()
    next_state = np.random.randn(STATE_DIM)
    done = np.random.choice([True, False])
    agent.store_transition(state, action, reward, next_state, done)

print(f"✅ Buffer rempli avec {len(agent.replay_buffer)} transitions.")

# Test de la mise à jour
update_info = agent.update(batch_size=32)
if update_info:
    print("✅ Mise à jour réussie:")
    print(f"   - Perte du critique: {update_info['critic_loss']:.4f}")
    print(f"   - Perte de l'acteur: {update_info['actor_loss']:.4f}")
    print(f"   - Alpha: {update_info['alpha']:.4f}")
else:
    print("❌ Échec de la mise à jour.")

# === CELLULE 8: TEST DE SAUVEGARDE/CHARGEMENT ===
print("\n" + "="*60)
print("🧪 TEST 5: SAUVEGARDE ET CHARGEMENT")
print("="*60)

# Sauvegarde
test_model_path = "../models/test_agent.pth"
agent.save(test_model_path)
print(f"✅ Modèle sauvegardé: {test_model_path}")

# Création d'un nouvel agent et chargement
new_agent = PortfolioSACAgent(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    k_assets=K_ASSETS,
    n_total_assets=N_TOTAL_ASSETS,
    n_indicators=N_INDICATORS,
    n_fundamentals=N_FUNDAMENTALS,
    device=device
)

new_agent.load(test_model_path)
print("✅ Modèle chargé avec succès.")

# Vérification que les poids sont identiques
dummy_state = np.random.randn(STATE_DIM)
action_original = agent.select_action(dummy_state, deterministic=True)
action_loaded = new_agent.select_action(dummy_state, deterministic=True)

if np.allclose(action_original, action_loaded, atol=1e-6):
    print("✅ Les actions sont identiques après sauvegarde/chargement.")
else:
    print("❌ Les actions diffèrent après sauvegarde/chargement.")

# === CELLULE 9: RÉSUMÉ ===
print("\n" + "="*60)
print("📝 RÉSUMÉ DES TESTS")
print("="*60)

print("""
✅ Tous les tests ont été exécutés avec succès:
1. Validation de l'architecture
2. Sélection d'action (stochastique/déterministe)
3. Replay buffer (ajout/échantillonnage)
4. Mise à jour de l'agent
5. Sauvegarde et chargement
""")

print("\n✅✅✅ VALIDATION DE L'AGENT TERMINÉE! ✅✅✅")


/opt/miniconda3/envs/finrl_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Initialisation du banc d'essai pour PortfolioSACAgent ---
✅ Importations réussies.

--- Définition des paramètres de test ---

Paramètres calculés:
- Nombre total d'actifs: 45
- Nombre d'indicateurs: 20
- Nombre de fondamentaux: 16
- Dimension de l'état: 497
- Dimension de l'action: 10


--- Création de l'agent SAC ---
Utilisation du device: mps
🤖 Agent SAC initialisé sur mps.
✅ Agent SAC créé avec succès!

🧪 TEST 1: VALIDATION DE L'ARCHITECTURE

🔍 Validation de l'architecture de l'agent...
  ✅ Acteur: Action shape torch.Size([1, 10]), Log-prob shape torch.Size([1, 1])
  ✅ Critique: Q1 shape torch.Size([1, 1]), Q2 shape torch.Size([1, 1])
❌ Erreur de validation: shape '[1, 10, 20]' is invalid for input of size 0
❌ Échec de la validation de l'architecture.

🧪 TEST 2: SÉLECTION D'ACTION

Action stochastique:
   - Shape: (10,)
   - Somme: 1.000000
   - Exemple: [0.02  0.054 0.107 0.047 0.322 0.117 0.05  0.112 0.088 0.084]

Action déterministe:
   - Shape: (10,)
   - Somme: 1.000000
  